# ARI711S – Group Project | Part 2: Hospital Shift Scheduler
## Merged Notebook – Juliette & Rejoice

---

### Overview
This notebook implements the full CSP pipeline for the Hospital Shift Scheduling problem:

| Section | Student | Responsibility |
|---|---|---|
| **Part A** | Juliette | Node Consistency & AC-3 Algorithm |
| **Part B** | Rejoice | MRV Heuristic & Backtracking Search |

---

### Problem Formulation (CSP)

| CSP Element | Meaning |
|---|---|
| **Variables (X)** | 21 weekly shift slots (7 days × 3 shifts) |
| **Domain (D)** | Set of available nurses per shift |
| **Constraints (C)** | Unary: leave days · Binary: Night→Morning rest · Higher-Order: max 5 shifts |

### The Three Constraints
- **Unary**: A nurse cannot work on a day they have pre-approved leave.
- **Binary**: A nurse on a Night shift cannot work the Morning of the very next day.
- **Higher-Order**: No nurse works more than 5 shifts in the week.

---
# PART A – Juliette: Node Consistency & AC-3
---

In [ ]:
from collections import deque

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
SHIFTS = ["Morning", "Afternoon", "Night"]

ALL_VARIABLES = [f"{day}_{shift}" for day in DAYS for shift in SHIFTS]

print(f"Total shift variables: {len(ALL_VARIABLES)}")
for v in ALL_VARIABLES:
    print(f"  {v}")

---
## Step 1 – Load Staff Data

We load nurse names and their leave days from a text file.
Each line has a nurse name followed by any days they are on leave.

In [ ]:
def load_staff(filepath):
    """
    Load nurse data from a staff text file.
    Format: NurseName   LeaveDay1   LeaveDay2 ...
    Returns: nurses (list), leave (dict)
    """
    nurses = []
    leave = {}

    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = [p.strip() for p in line.replace(",", "\t").split("\t") if p.strip()]
            if not parts:
                continue
            name = parts[0]
            nurses.append(name)
            leave_days = {token for token in parts[1:] if token in DAYS}
            leave[name] = leave_days

    return nurses, leave

print("load_staff() defined.")

---
## Step 2 – The Shift_AI_Solver Class

The solver class holds all domains and constraints.
Juliette's methods handle the preparation phase before backtracking:
- `enforce_node_consistency()` — unary constraint pruning
- `revise(x, y)` — binary arc consistency  
- `ac3()` — full AC-3 propagation

Rejoice's methods handle the search phase:
- `select_unassigned_variable()` — MRV heuristic
- `forward_check()` — forward checking after assignment
- `backtrack()` — recursive backtracking search

In [ ]:
class Shift_AI_Solver:
    def __init__(self, nurses, leave):
        self.nurses = nurses
        self.leave = leave
        self.variables = ALL_VARIABLES[:]
        self.domains = {var: set(nurses) for var in self.variables}

    # ------------------------------------------------------------------
    # JULIETTE'S METHODS – Constraint Propagation
    # ------------------------------------------------------------------

    def enforce_node_consistency(self):
        """Unary constraint: remove nurses on leave from shift domains."""
        print("Enforcing node consistency (unary constraints)...")
        for variable in self.variables:
            day = variable.split("_")[0]
            to_remove = {
                nurse for nurse in self.domains[variable]
                if day in self.leave.get(nurse, set())
            }
            self.domains[variable] -= to_remove
            if to_remove:
                print(f"  [{variable}] Removed (on leave): {to_remove}")
        print("Node consistency enforced.\n")

    def revise(self, x, y):
        """Binary arc consistency: Night shift -> next day Morning rest rule."""
        x_day, x_type = x.rsplit("_", 1)
        y_day, y_type = y.rsplit("_", 1)

        if x_type != "Night" or y_type != "Morning":
            return False
        if DAYS.index(y_day) != DAYS.index(x_day) + 1:
            return False

        revised = False
        to_remove = set()

        for nurse in self.domains[x]:
            remaining_for_y = self.domains[y] - {nurse}
            if len(remaining_for_y) == 0:
                to_remove.add(nurse)
                revised = True

        self.domains[x] -= to_remove
        return revised

    def ac3(self):
        """AC-3: propagate arc consistency across all Night->Morning arc pairs."""
        print("Running AC-3 algorithm...")
        queue = deque()
        for i in range(len(DAYS) - 1):
            night = f"{DAYS[i]}_Night"
            morning = f"{DAYS[i+1]}_Morning"
            queue.append((night, morning))

        print(f"  Initial arc queue: {len(queue)} arcs")

        while queue:
            x, y = queue.popleft()
            if self.revise(x, y):
                print(f"  Revised [{x}] → remaining domain: {self.domains[x]}")
                if len(self.domains[x]) == 0:
                    print(f"  FAILURE: [{x}] domain is empty!")
                    return False
                x_day, x_type = x.rsplit("_", 1)
                if x_type == "Night":
                    x_idx = DAYS.index(x_day)
                    if x_idx > 0:
                        prev_night = f"{DAYS[x_idx - 1]}_Night"
                        queue.append((prev_night, x))

        print("AC-3 complete. All arcs are consistent.\n")
        return True

    def print_domains(self):
        print("\nDomain Summary:")
        for var in self.variables:
            nurses_list = sorted(self.domains[var])
            print(f"  {var:<25} ({len(nurses_list)} nurses): {nurses_list}")

    # ------------------------------------------------------------------
    # REJOICE'S METHODS – MRV Heuristic & Backtracking
    # ------------------------------------------------------------------

    def select_unassigned_variable(self, assignment, domains):
        """
        MRV (Minimum Remaining Values) heuristic.
        Selects the unassigned variable with the fewest values in its domain.
        Ties are broken by the variable that appears earliest in the list.
        """
        unassigned = [v for v in self.variables if v not in assignment]
        # MRV: pick variable with smallest remaining domain
        return min(unassigned, key=lambda var: len(domains[var]))

    def is_consistent(self, var, nurse, assignment, domains):
        """
        Check all constraints for assigning `nurse` to `var`:
          1. Unary: nurse must be in current domain (leave already pruned)
          2. Binary: night shift -> cannot work next morning
          3. Higher-order: nurse cannot exceed 5 shifts in the week
        """
        # Unary: nurse must still be in domain
        if nurse not in domains[var]:
            return False

        # Binary: Night -> Morning rest constraint
        var_day, var_type = var.rsplit("_", 1)
        var_idx = DAYS.index(var_day)

        if var_type == "Night":
            # Nurse cannot work next day's Morning
            if var_idx < len(DAYS) - 1:
                next_morning = f"{DAYS[var_idx + 1]}_Morning"
                if assignment.get(next_morning) == nurse:
                    return False

        if var_type == "Morning":
            # Nurse cannot have worked prev day's Night
            if var_idx > 0:
                prev_night = f"{DAYS[var_idx - 1]}_Night"
                if assignment.get(prev_night) == nurse:
                    return False

        # Higher-order: max 5 shifts per nurse
        current_shifts = sum(1 for v, n in assignment.items() if n == nurse)
        if current_shifts >= 5:
            return False

        return True

    def forward_check(self, var, nurse, domains):
        """
        Forward checking: after assigning `nurse` to `var`, prune domains
        of future variables to maintain arc consistency.
        Returns a dict of {variable: removed_values} for undo on backtrack,
        or None if a domain becomes empty (failure detected early).
        """
        pruned = {}
        var_day, var_type = var.rsplit("_", 1)
        var_idx = DAYS.index(var_day)

        # Binary: if Night assigned, remove nurse from next Morning domain
        if var_type == "Night" and var_idx < len(DAYS) - 1:
            next_morning = f"{DAYS[var_idx + 1]}_Morning"
            if nurse in domains[next_morning]:
                pruned.setdefault(next_morning, set()).add(nurse)
                domains[next_morning] -= {nurse}
                if len(domains[next_morning]) == 0:
                    return None  # domain wipeout

        return pruned

    def backtrack(self, assignment=None, domains=None, depth=0):
        """
        Recursive backtracking search with MRV + forward checking.
        Returns a complete assignment dict {shift_variable: nurse_name},
        or None if no solution exists.
        """
        if assignment is None:
            assignment = {}
        if domains is None:
            # Deep copy domains so backtracking can restore them
            domains = {var: set(vals) for var, vals in self.domains.items()}

        # Base case: all variables assigned
        if len(assignment) == len(self.variables):
            return assignment

        # MRV: pick best unassigned variable
        var = self.select_unassigned_variable(assignment, domains)

        for nurse in sorted(domains[var]):  # sorted for determinism
            if self.is_consistent(var, nurse, assignment, domains):
                assignment[var] = nurse

                # Forward checking
                pruned = self.forward_check(var, nurse, domains)

                if pruned is not None:  # No domain wipeout
                    result = self.backtrack(assignment, domains, depth + 1)
                    if result is not None:
                        return result

                # Undo assignment and restore pruned values
                del assignment[var]
                if pruned:
                    for pruned_var, removed in pruned.items():
                        domains[pruned_var] |= removed

        return None  # trigger backtrack

    def print_schedule(self, assignment):
        """Pretty-print the final schedule as a weekly grid."""
        print("\n" + "=" * 60)
        print("FINAL SCHEDULE")
        print("=" * 60)
        print(f"{'Day':<12} {'Morning':<22} {'Afternoon':<22} {'Night'}")
        print("-" * 80)
        for day in DAYS:
            m = assignment.get(f"{day}_Morning", "UNASSIGNED")
            a = assignment.get(f"{day}_Afternoon", "UNASSIGNED")
            n = assignment.get(f"{day}_Night", "UNASSIGNED")
            print(f"  {day:<10} {m:<22} {a:<22} {n}")
        print("=" * 60)

print("Shift_AI_Solver class defined successfully (merged).")

---
## Step 3 – Create Test Data & Run Juliette's Phase

We create sample staff files and run node consistency + AC-3.

In [ ]:
# staff_small.txt (Juliette's original test data)
sample_staff_small = """# staff_small.txt
Kamati R\tMonday
Swartbooi I\tTuesday
Mudge D
Naruseb J\tWednesday
Eigowab S\tFriday
Beukes D\tSaturday
Kahuure K
Hanse-Himarwa K\tThursday
Mwandingi F\tSunday
Iivula-Ithana P
Gaweseb T\tFriday
Kambonde A\tMonday\tWednesday
"""
with open("staff_small.txt", "w") as f:
    f.write(sample_staff_small)
print("staff_small.txt created.")

In [ ]:
# staff_medium.txt (Rejoice's test data – 15 nurses)
sample_staff_medium = """# staff_medium.txt – 15 nurses
Kamati R\tMonday
Swartbooi I\tTuesday
Mudge D
Naruseb J\tWednesday
Eigowab S\tFriday
Beukes D\tSaturday
Kahuure K
Hanse-Himarwa K\tThursday
Mwandingi F\tSunday
Iivula-Ithana P
Gaweseb T\tFriday
Kambonde A\tMonday\tWednesday
Nakale B\tTuesday
Amweelo C\tSaturday
Iipinge D
"""
with open("staff_medium.txt", "w") as f:
    f.write(sample_staff_medium)
print("staff_medium.txt created.")

In [ ]:
# staff_complex.txt (Rejoice's harder test – many leave days)
sample_staff_complex = """# staff_complex.txt – constrained scenario
Kamati R\tMonday\tTuesday
Swartbooi I\tTuesday\tWednesday
Mudge D\tWednesday\tThursday
Naruseb J\tThursday\tFriday
Eigowab S\tFriday\tSaturday
Beukes D\tSaturday\tSunday
Kahuure K\tMonday
Hanse-Himarwa K\tThursday
Mwandingi F\tSunday
Iivula-Ithana P\tFriday
Gaweseb T\tMonday\tFriday
Kambonde A\tMonday\tWednesday\tFriday
Nakale B\tTuesday\tSaturday
Amweelo C\tSaturday\tSunday
Iipinge D\tWednesday
"""
with open("staff_complex.txt", "w") as f:
    f.write(sample_staff_complex)
print("staff_complex.txt created.")

In [ ]:
nurses, leave = load_staff("staff_small.txt")

print(f"Loaded {len(nurses)} nurses:\n")
for name in nurses:
    leave_days = leave[name] if leave[name] else {"None"}
    print(f"  {name:<22} Leave: {leave_days}")

### 3a – enforce_node_consistency()

Applies the **unary constraint**: nurses on leave are removed from 
all shift domains for that day.

In [ ]:
solver = Shift_AI_Solver(nurses, leave)

print("BEFORE node consistency:")
print(f"  Friday_Morning domain size: {len(solver.domains['Friday_Morning'])}")
print(f"  Monday_Night domain size:   {len(solver.domains['Monday_Night'])}")
print()

solver.enforce_node_consistency()

print("AFTER node consistency:")
print(f"  Friday_Morning: {sorted(solver.domains['Friday_Morning'])}")
print(f"  Monday_Night:   {sorted(solver.domains['Monday_Night'])}")

### 3b – ac3()

Applies the **binary constraint** (Night → next Morning rest rule) 
through AC-3 propagation across all 6 Night→Morning arc pairs.

In [ ]:
success = solver.ac3()

if success:
    print("✅ AC-3 succeeded. Domains are arc-consistent.")
else:
    print("❌ AC-3 failed. No valid schedule possible.")

In [ ]:
solver.print_domains()

---
## Step 4 – Unit Tests for revise() (Juliette)

We test the binary arc consistency logic in isolation.

In [ ]:
print("=== Test 1: Nurse removed when sole option for next Morning ===")
t = Shift_AI_Solver(["Nurse_A", "Nurse_B"], {})
t.domains["Tuesday_Morning"] = {"Nurse_A"}
t.domains["Monday_Night"] = {"Nurse_A", "Nurse_B"}
changed = t.revise("Monday_Night", "Tuesday_Morning")
assert changed == True
assert "Nurse_A" not in t.domains["Monday_Night"]
print(f"  Monday_Night after revise: {t.domains['Monday_Night']}")
print("  ✅ PASSED\n")

print("=== Test 2: No change for non-binary-constraint pairs ===")
t2 = Shift_AI_Solver(["Nurse_A", "Nurse_B"], {})
result = t2.revise("Monday_Morning", "Monday_Afternoon")
assert result == False
print(f"  Result: {result} — correctly False")
print("  ✅ PASSED\n")

print("=== Test 3: No change for non-consecutive days ===")
t3 = Shift_AI_Solver(["Nurse_A", "Nurse_B"], {})
result = t3.revise("Monday_Night", "Wednesday_Morning")
assert result == False
print(f"  Result: {result} — correctly False")
print("  ✅ PASSED")

---
# PART B – Rejoice: MRV Heuristic & Backtracking Search
---

### Overview
After Juliette's constraint propagation prunes the domains, this section
implements the **backtracking search** to find a valid complete assignment.

| Method | Role |
|---|---|
| `select_unassigned_variable()` | MRV: pick variable with fewest options |
| `is_consistent()` | Check all three constraints before assigning |
| `forward_check()` | Prune future domains immediately after assignment |
| `backtrack()` | Recursive search with undo on failure |

### Why MRV?
The **Minimum Remaining Values** heuristic selects the variable whose 
domain is most constrained. This:
- Detects failures early (fail-first principle)
- Reduces branching by tackling hard decisions first
- Combined with forward checking, avoids many dead-end paths

---
## Step 5 – Unit Tests for select_unassigned_variable() (Rejoice)

We verify the MRV heuristic selects the most constrained variable.

In [ ]:
print("=== MRV Test 1: Picks variable with smallest domain ===")
t_mrv = Shift_AI_Solver(["Nurse_A", "Nurse_B", "Nurse_C"], {})
# Manually shrink one domain to simulate a tightly constrained shift
test_domains = {var: set(["Nurse_A", "Nurse_B", "Nurse_C"]) for var in ALL_VARIABLES}
test_domains["Wednesday_Night"] = {"Nurse_A"}  # smallest domain
assignment = {v: "Nurse_A" for v in ALL_VARIABLES[:5]}  # pretend some are assigned
# Only test on unassigned; remove Wednesday_Night from the fake assigned set
assignment = {}
selected = t_mrv.select_unassigned_variable(assignment, test_domains)
assert selected == "Wednesday_Night", f"Expected Wednesday_Night, got {selected}"
print(f"  MRV selected: {selected}")
print("  ✅ PASSED\n")

print("=== MRV Test 2: Skips already-assigned variables ===")
t_mrv2 = Shift_AI_Solver(["Nurse_A", "Nurse_B"], {})
test_domains2 = {var: {"Nurse_A", "Nurse_B"} for var in ALL_VARIABLES}
test_domains2["Monday_Morning"] = {"Nurse_A"}  # smallest but already assigned
assignment2 = {"Monday_Morning": "Nurse_A"}
selected2 = t_mrv2.select_unassigned_variable(assignment2, test_domains2)
assert selected2 != "Monday_Morning"
print(f"  MRV correctly skipped assigned var, selected: {selected2}")
print("  ✅ PASSED")

---
## Step 6 – Unit Tests for is_consistent() (Rejoice)

Verifies constraint checking before any assignment is made.

In [ ]:
print("=== Consistency Test 1: Night → Morning binary constraint ===")
tc = Shift_AI_Solver(["Nurse_A", "Nurse_B"], {})
tc_domains = {var: {"Nurse_A", "Nurse_B"} for var in ALL_VARIABLES}
assignment_c = {"Monday_Night": "Nurse_A"}
# Nurse_A cannot work Tuesday_Morning after Monday_Night
result = tc.is_consistent("Tuesday_Morning", "Nurse_A", assignment_c, tc_domains)
assert result == False, "Should be inconsistent – night/morning clash"
print(f"  Nurse_A for Tuesday_Morning (after Monday_Night): {result} — correctly False")
result2 = tc.is_consistent("Tuesday_Morning", "Nurse_B", assignment_c, tc_domains)
assert result2 == True
print(f"  Nurse_B for Tuesday_Morning: {result2} — correctly True")
print("  ✅ PASSED\n")

print("=== Consistency Test 2: Max 5 shifts higher-order constraint ===")
tc2 = Shift_AI_Solver(["Nurse_A", "Nurse_B"], {})
tc2_domains = {var: {"Nurse_A", "Nurse_B"} for var in ALL_VARIABLES}
# Assign Nurse_A to 5 shifts already
assignment_5 = {
    "Monday_Morning": "Nurse_A",
    "Monday_Afternoon": "Nurse_A",
    "Tuesday_Afternoon": "Nurse_A",
    "Wednesday_Afternoon": "Nurse_A",
    "Thursday_Afternoon": "Nurse_A",
}
result3 = tc2.is_consistent("Friday_Morning", "Nurse_A", assignment_5, tc2_domains)
assert result3 == False, "Should be False — max 5 shifts exceeded"
print(f"  Nurse_A for 6th shift: {result3} — correctly False (max 5 exceeded)")
print("  ✅ PASSED")

---
## Step 7 – Full Pipeline: staff_medium.txt (Rejoice)

Run the complete pipeline: load → node consistency → AC-3 → backtracking.

In [ ]:
print("=" * 60)
print("PIPELINE: staff_medium.txt")
print("=" * 60)

nurses_m, leave_m = load_staff("staff_medium.txt")
print(f"Loaded {len(nurses_m)} nurses.")

solver_m = Shift_AI_Solver(nurses_m, leave_m)
solver_m.enforce_node_consistency()
ac3_ok = solver_m.ac3()

if not ac3_ok:
    print("❌ AC-3 failed. Cannot proceed to backtracking.")
else:
    print("Running backtracking search (MRV + forward checking)...")
    solution_m = solver_m.backtrack()

    if solution_m:
        print(f"✅ Solution found! ({len(solution_m)} shifts assigned)")
        solver_m.print_schedule(solution_m)
    else:
        print("❌ No valid schedule found.")

---
## Step 8 – Full Pipeline: staff_complex.txt (Rejoice)

Tests a harder scenario where nurses have many leave days, 
putting more pressure on the MRV + backtracking to find a valid schedule.

In [ ]:
print("=" * 60)
print("PIPELINE: staff_complex.txt")
print("=" * 60)

nurses_c, leave_c = load_staff("staff_complex.txt")
print(f"Loaded {len(nurses_c)} nurses.")

solver_c = Shift_AI_Solver(nurses_c, leave_c)
solver_c.enforce_node_consistency()
ac3_ok_c = solver_c.ac3()

if not ac3_ok_c:
    print("❌ AC-3 failed. Cannot proceed to backtracking.")
else:
    print("Running backtracking search (MRV + forward checking)...")
    solution_c = solver_c.backtrack()

    if solution_c:
        print(f"✅ Solution found! ({len(solution_c)} shifts assigned)")
        solver_c.print_schedule(solution_c)

        # Verify higher-order constraint
        print("\nShift counts per nurse:")
        from collections import Counter
        counts = Counter(solution_c.values())
        for nurse, count in sorted(counts.items()):
            flag = "⚠️ OVER LIMIT" if count > 5 else ""
            print(f"  {nurse:<22} {count} shifts {flag}")
    else:
        print("❌ No valid schedule found.")

---
## Step 9 – How Constraints Are Satisfied (Rejoice)

This section documents how each constraint is enforced in the pipeline.

### Constraint Satisfaction Walkthrough

| Constraint | Where Enforced | How |
|---|---|---|
| **Unary** – nurse on leave | `enforce_node_consistency()` | Removes nurse from all domain sets for that day before search begins |
| **Binary** – Night→Morning rest | `ac3()` + `is_consistent()` + `forward_check()` | AC-3 prunes domains pre-search; `is_consistent` checks live assignment; `forward_check` prunes immediately after each assignment |
| **Higher-Order** – max 5 shifts | `is_consistent()` | Counts nurse's current shifts in assignment; rejects if already at 5 |

### Why Forward Checking Helps
After assigning a nurse to a Night shift, `forward_check()` immediately 
removes that nurse from the *next day's Morning* domain. This means:
- The violation is caught **before** we recurse deeper
- If the Morning domain becomes empty, we backtrack **immediately** 
  (domain wipeout detection) instead of discovering the failure later

In [ ]:
print("=== Forward Checking Test ===")
tf = Shift_AI_Solver(["Nurse_A", "Nurse_B"], {})
tf_domains = {var: {"Nurse_A", "Nurse_B"} for var in ALL_VARIABLES}

print(f"  Before: Tuesday_Morning domain = {tf_domains['Tuesday_Morning']}")
pruned = tf.forward_check("Monday_Night", "Nurse_A", tf_domains)
print(f"  After assigning Nurse_A to Monday_Night:")
print(f"  Tuesday_Morning domain = {tf_domains['Tuesday_Morning']}")
print(f"  Pruned: {pruned}")
assert "Nurse_A" not in tf_domains["Tuesday_Morning"]
print("  ✅ PASSED – forward checking correctly pruned next Morning")

---
## Summary

### Part A – Juliette (Constraint Propagation)

| Method | Constraint Type | Effect |
|---|---|---|
| `enforce_node_consistency()` | Unary | Removed nurses on leave from shift domains |
| `revise(x, y)` | Binary | Removed nurses from Night domains where they were the sole option for next Morning |
| `ac3()` | Binary (propagated) | Applied revise() across all 6 Night→Morning arcs with ripple-effect re-checking |

### Part B – Rejoice (Search)

| Method | Role | Effect |
|---|---|---|
| `select_unassigned_variable()` | MRV Heuristic | Always chooses most constrained shift first to detect failures early |
| `is_consistent()` | Constraint check | Validates unary, binary, and higher-order constraints before every assignment |
| `forward_check()` | Domain pruning | Immediately removes nurse from next Morning domain after Night assignment |
| `backtrack()` | Search | Recursively assigns nurses with full undo support on failure |

After both phases, the solver produces a complete, constraint-satisfying 
weekly nurse schedule for all 21 shift slots.